# V2: Model Creation
## 4.1 Loading Files

In [1]:
import pandas as pd

train_v2_numericalized = pd.read_parquet('data/processed/train_v2_numericalized.parquet')
validation_v2_numericalized = pd.read_parquet('data/processed/validation_v2_numericalized.parquet')

print(f'Train Shape: {train_v2_numericalized.shape}')
print(f'Validation Shape:{validation_v2_numericalized.shape}')

assert 'tokens_ids' in train_v2_numericalized.columns
assert 'tokens_ids' in validation_v2_numericalized

for index in [1, 10, 200]:
    sequence = train_v2_numericalized['tokens_ids'].iloc[index]
    print(f'Sequence of tokens turned into ids: {sequence}')
    print(f'Number of tokens: {len(sequence)}\n')

print(f'Train Columns: {train_v2_numericalized.columns.tolist()}')
print(f'Validation Columns: {validation_v2_numericalized.columns.tolist()}')

Train Shape: (22809, 4)
Validation Shape:(5703, 4)
Sequence of tokens turned into ids: [248 249 337 161 388 258 662 154 592 161 588 154 155 155 247 249 337 161
 388 258 662 154 592 161 588 154 155 164 850 154 155 155]
Number of tokens: 32

Sequence of tokens turned into ids: [247 249 525 154 155]
Number of tokens: 5

Sequence of tokens turned into ids: [247 249 760 154 367 155]
Number of tokens: 6

Train Columns: ['diff', 'top_level_label', 'tokens', 'tokens_ids']
Validation Columns: ['diff', 'top_level_label', 'tokens', 'tokens_ids']


## 4.2 Label to ID

In [2]:
train_unique_labels = train_v2_numericalized['top_level_label'].unique()
validation_unique_labels = validation_v2_numericalized['top_level_label'].unique()
assert not set(validation_unique_labels).difference(set(train_unique_labels))

label_to_ids = {label: label_id for label_id, label in enumerate(sorted(train_unique_labels))}

In [3]:
train_v2_numericalized['label_id'] = train_v2_numericalized['top_level_label'].map(label_to_ids)
assert not train_v2_numericalized['label_id'].isnull().any()

validation_v2_numericalized['label_id'] = validation_v2_numericalized['top_level_label'].map(label_to_ids)
assert not validation_v2_numericalized['label_id'].isnull().any()


for index in [1, 20, 1000]:
    label = train_v2_numericalized['top_level_label'].loc[index]
    label_id = train_v2_numericalized['label_id'].loc[index]
    print(f'Label: {label}\nLabel ID: {label_id}')

Label: call
Label ID: 1
Label: control_flow
Label ID: 2
Label: expression
Label ID: 3


## 4.3 Saving the Label to Id

In [4]:
import json
train_v2_numericalized.to_parquet('data/processed/train_v2_label_id.parquet', index=False)
validation_v2_numericalized.to_parquet('data/processed/validation_v2_label_id.parquet', index=False)

ids_to_labels = {ident: label for label, ident in label_to_ids.items()}
print(ids_to_labels)

with open('data/processed/label_to_id.json', 'w') as file:
    json.dump(label_to_ids, file)

with open('data/processed/id_to_label.json', 'w') as file:
    json.dump(ids_to_labels, file)


{0: 'assignment', 1: 'call', 2: 'control_flow', 3: 'expression', 4: 'identifier'}


## 4.4 Creating the Architecture for the Dataset

In [5]:
import torch
from torch.utils.data import Dataset

class BugFixDataset(Dataset):
    def __init__(self, dataframe):
        super().__init__()
        self.dataframe = dataframe
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, index):
        tokens_in_ids = self.dataframe['tokens_ids'].iloc[index]
        label_id = self.dataframe['label_id'].iloc[index]
        return torch.tensor(tokens_in_ids, dtype=torch.long), torch.tensor(label_id, dtype=torch.long)

train_dataset = BugFixDataset(train_v2_numericalized)
validation_dataset = BugFixDataset(validation_v2_numericalized)

In [6]:
tokens, label = train_dataset[0]
print(tokens)
print(tokens.dtype, tokens.shape)
print(label)
print(label.dtype, label.shape)

tensor([248, 249, 542, 576, 161, 603, 583, 521, 154, 602, 164, 745, 154, 771,
        161, 211, 155, 155, 242, 247, 249, 542, 576, 161, 603, 583, 521, 154,
        602, 164, 745, 154, 771, 155, 155, 242])
torch.int64 torch.Size([36])
tensor(1)
torch.int64 torch.Size([])


In [7]:
for index in [0, 10, 200]:
    train_tokens, train_label_id = train_dataset[index]
    assert train_tokens.numel() == len(train_v2_numericalized['tokens_ids'].iloc[index])
    assert train_label_id.item() == train_v2_numericalized['label_id'].iloc[index]

    validation_tokens, validation_label_id = validation_dataset[index]
    assert validation_tokens.numel() == len(validation_v2_numericalized['tokens_ids'].iloc[index])
    assert validation_label_id.item() == validation_v2_numericalized['label_id'].iloc[index]

assert len(train_dataset) == len(train_v2_numericalized)
assert len(validation_dataset) == len(validation_v2_numericalized)

In [8]:
def my_collate_fn(batch): #batch has tensors for tokens, labels
    token_tensors, label_tensors = zip(*batch)
    batch_max_length = 0
    for tokens in token_tensors:
        batch_max_length = max(batch_max_length, len(tokens))

    padded_token_tensors = []
    for tokens in token_tensors:
        amount_of_padding = batch_max_length - len(tokens)
        pads = torch.zeros(amount_of_padding, dtype=torch.long, device=tokens.device)
        padded_sequence = torch.cat((tokens, pads))
        assert torch.equal(padded_sequence[:len(tokens)], tokens)
        padded_token_tensors.append(padded_sequence)   

    stacked_padded_tokens = torch.stack(padded_token_tensors)
    labels = torch.stack(label_tensors)

    padding_mask = stacked_padded_tokens == 0

    return stacked_padded_tokens, labels, padding_mask

In [9]:
test_batch = [train_dataset[0], train_dataset[10], train_dataset[200]]
tokens, labels, mask = my_collate_fn(test_batch)

for row_index, (original_tokens, _) in enumerate(test_batch):
    original_length = len(original_tokens)

    assert torch.equal(tokens[row_index, :original_length], original_tokens) #first argumennt is row and second is columns
    assert (tokens[row_index, original_length:] == 0).all()

In [10]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=my_collate_fn)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False, collate_fn=my_collate_fn)

train_tokens, train_labels, train_mask = next(iter(train_loader))
validation_tokens, validation_labels, validation_mask = next(iter(validation_loader))

assert train_tokens.shape == train_mask.shape
assert validation_tokens.shape == validation_mask.shape

assert train_tokens.shape[0] == train_labels.shape[0]
assert validation_tokens.shape[0] == validation_labels.shape[0]

assert train_tokens.dtype == torch.long
assert validation_tokens.dtype == torch.long

assert train_labels.dtype == torch.long
assert validation_labels.dtype == torch.long

assert train_mask.dtype == torch.bool
assert validation_mask.dtype == torch.bool

print(train_tokens.shape)
print(train_labels.shape)
print(train_mask.shape)

print(validation_tokens.shape)
print(validation_labels.shape)
print(validation_mask.shape)

torch.Size([32, 86])
torch.Size([32])
torch.Size([32, 86])
torch.Size([32, 76])
torch.Size([32])
torch.Size([32, 76])


In [11]:
import numpy as np

train_lengths = [len(tokens) for tokens, _ in train_dataset]
print(min(train_lengths))
print(max(train_lengths))
print(sum(train_lengths) / len(train_lengths))

print(np.percentile(train_lengths, [25, 50, 75, 90, 95, 99]))

3
554
25.529220921566047
[  7.  21.  34.  52.  66. 112.]


In [12]:
import torch.nn as nn
embedding = nn.Embedding(num_embeddings=947, embedding_dim=128, padding_idx=0)
embedded_tokens = embedding(train_tokens)

print(train_tokens.shape)
print(embedded_tokens.shape)

torch.Size([32, 86])
torch.Size([32, 86, 128])


In [13]:
real_token_mask = (~train_mask).int().unsqueeze(2)
print(train_mask.shape)
print(real_token_mask.shape)
print(train_mask[0])
print(real_token_mask[0])

torch.Size([32, 86])
torch.Size([32, 86, 1])
tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True])
tensor([[1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [1],
        [0],
       

In [14]:
masked_embeddings = embedded_tokens * real_token_mask
print(embedded_tokens.shape)
print(real_token_mask.shape)
print(masked_embeddings.shape)

token_sums = masked_embeddings.sum(dim=1)
mean_pooling = token_sums / real_token_mask.sum(dim=1)
print(token_sums.shape)
print(real_token_mask.sum(dim=1).shape)
print(mean_pooling.shape)
print(torch.isnan(mean_pooling).any())
print(torch.isinf(mean_pooling).any())

torch.Size([32, 86, 128])
torch.Size([32, 86, 1])
torch.Size([32, 86, 128])
torch.Size([32, 128])
torch.Size([32, 1])
torch.Size([32, 128])
tensor(False)
tensor(False)


In [15]:
classifier = nn.Linear(128, 5)
logits = classifier(mean_pooling)
print(logits.shape)

torch.Size([32, 5])


In [16]:
class BugFixClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embeddings = nn.Embedding(num_embeddings=947, embedding_dim=128, padding_idx=0)
        self.classifier = nn.Linear(128,5)

    def forward(self, tokens, padding_mask):
        embedded_tokens = self.embeddings(tokens) #(batchsize, maxlen, embeddingdim)
        real_token_mask = (~padding_mask).int() #(batchsize, maxlen) results in 1s/0s each diff has a vector of 1s/0s
        unsqueezed = real_token_mask.unsqueeze(dim=2) #(batchsize, maxlen, 1)
        masked = embedded_tokens * unsqueezed #(batchsize, maxlen, embeddingdim)

        token_sums = masked.sum(dim=1)
        token_counts = unsqueezed.sum(dim=1)

        mean_pooling = token_sums / token_counts
        logits = self.classifier(mean_pooling)
        return logits

In [17]:
model = BugFixClassifier()
logits = model(train_tokens, train_mask)
print(logits.shape)

torch.Size([32, 5])


In [18]:
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, train_labels)
print(loss)
print(loss.shape)

tensor(1.6675, grad_fn=<NllLossBackward0>)
torch.Size([])


In [19]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad()

logits = model(train_tokens, train_mask)
loss = criterion(logits, train_labels)

loss.backward()
optimizer.step()

print(loss.item())

1.667513132095337


In [20]:
total_loss = 0
total_correct = 0
total_examples = 0

for tokens, labels, padding_mask in train_loader:
    optimizer.zero_grad()

    logits = model(tokens, padding_mask)
    loss = criterion(logits, labels)
    
    loss.backward()
    optimizer.step()
    predictions = logits.argmax(dim=1)
    total_correct += (predictions == labels).sum().item()
    total_examples += labels.size(0)
    total_loss += loss.item() * labels.size(0)

epoch_loss = total_loss / total_examples
epoch_accuracy = total_correct / total_examples

print(epoch_loss)
print(epoch_accuracy)

0.7846992709017985
0.6778903064579771


In [21]:
model.eval()

validation_loss = 0
validation_correct = 0
validation_examples = 0

with torch.no_grad():
    for tokens, labels, padding_mask in validation_loader:
        logits = model(tokens, padding_mask)
        loss = criterion(logits, labels)
        predictions = logits.argmax(dim=1)
        validation_loss += loss.item() * labels.size(0)
        validation_correct += (predictions == labels).sum().item()
        validation_examples += labels.size(0)

epoch_loss = validation_loss / validation_examples
epoch_accuracy = validation_correct / validation_examples

print(epoch_loss)
print(epoch_accuracy)

0.6195562982504856
0.7345256882342627


In [22]:
torch.manual_seed(42)

model = BugFixClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_validation_loss = float('inf')
for epoch in range(15):
    model.train()
    total_loss = 0
    total_correct = 0
    total_examples = 0
    for tokens, labels, padding_mask in train_loader:
        optimizer.zero_grad()

        logits = model(tokens, padding_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        predictions = logits.argmax(dim=1)
        total_correct += (predictions == labels).sum().item()
        total_examples += labels.size(0)
        total_loss += loss.item() * labels.size(0)

    train_loss = total_loss / total_examples
    train_accuracy = total_correct / total_examples

    model.eval()

    validation_loss = 0
    validation_correct = 0
    validation_examples = 0

    with torch.no_grad():
        for tokens, labels, padding_mask in validation_loader:
            logits = model(tokens, padding_mask)
            loss = criterion(logits, labels)
            predictions = logits.argmax(dim=1)
            validation_loss += loss.item() * labels.size(0)
            validation_correct += (predictions == labels).sum().item()
            validation_examples += labels.size(0)

    validation_epoch_loss = validation_loss / validation_examples
    validation_accuracy = validation_correct / validation_examples
    if validation_epoch_loss < best_validation_loss:
        best_validation_loss = validation_epoch_loss
        torch.save({'epoch': epoch + 1, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'validation_loss': validation_epoch_loss}, 'models/v2_best_model.pt')


    print(f'Epoch: {epoch + 1}, Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f} Validation Loss: {validation_epoch_loss:.4f} Validation Accuracy: {validation_accuracy:.4f}')

Epoch: 1, Train Loss: 0.7844, Train Accuracy: 0.6835 Validation Loss: 0.6197 Validation Accuracy: 0.7326
Epoch: 2, Train Loss: 0.6025, Train Accuracy: 0.7366 Validation Loss: 0.5769 Validation Accuracy: 0.7438
Epoch: 3, Train Loss: 0.5693, Train Accuracy: 0.7499 Validation Loss: 0.5580 Validation Accuracy: 0.7563
Epoch: 4, Train Loss: 0.5509, Train Accuracy: 0.7605 Validation Loss: 0.5467 Validation Accuracy: 0.7598
Epoch: 5, Train Loss: 0.5381, Train Accuracy: 0.7667 Validation Loss: 0.5411 Validation Accuracy: 0.7594
Epoch: 6, Train Loss: 0.5280, Train Accuracy: 0.7725 Validation Loss: 0.5363 Validation Accuracy: 0.7652
Epoch: 7, Train Loss: 0.5202, Train Accuracy: 0.7799 Validation Loss: 0.5321 Validation Accuracy: 0.7708
Epoch: 8, Train Loss: 0.5135, Train Accuracy: 0.7814 Validation Loss: 0.5317 Validation Accuracy: 0.7749
Epoch: 9, Train Loss: 0.5080, Train Accuracy: 0.7840 Validation Loss: 0.5304 Validation Accuracy: 0.7749
Epoch: 10, Train Loss: 0.5027, Train Accuracy: 0.7867 V